# U-Net Implementation for Leaf Disease Segmentation
### With Layer-by-Layer Feature Map Visualization


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models


## Generate Synthetic Leaf Image (Demo Purpose)

In [ ]:
img = np.zeros((256,256,3))
img[:,:,1] = 0.6

for i in range(256):
    for j in range(256):
        if (i-128)**2 + (j-128)**2 < 30**2:
            img[i,j] = [0.6,0.3,0.1]

plt.figure()
plt.imshow(img)
plt.title('Synthetic Leaf Image')
plt.axis('off')
plt.show()

## Build U-Net Model

In [ ]:
def unet_model():
    inputs = layers.Input((256,256,3))

    c1 = layers.Conv2D(32,(3,3),activation='relu',padding='same')(inputs)
    c1 = layers.Conv2D(32,(3,3),activation='relu',padding='same')(c1)
    p1 = layers.MaxPooling2D((2,2))(c1)

    c2 = layers.Conv2D(64,(3,3),activation='relu',padding='same')(p1)
    c2 = layers.Conv2D(64,(3,3),activation='relu',padding='same')(c2)
    p2 = layers.MaxPooling2D((2,2))(c2)

    b = layers.Conv2D(128,(3,3),activation='relu',padding='same')(p2)

    u1 = layers.UpSampling2D((2,2))(b)
    u1 = layers.concatenate([u1,c2])
    c3 = layers.Conv2D(64,(3,3),activation='relu',padding='same')(u1)

    u2 = layers.UpSampling2D((2,2))(c3)
    u2 = layers.concatenate([u2,c1])
    c4 = layers.Conv2D(32,(3,3),activation='relu',padding='same')(u2)

    outputs = layers.Conv2D(1,(1,1),activation='sigmoid')(c4)

    return models.Model(inputs, outputs)

model = unet_model()
model.summary()

## Forward Pass (Visualization)

In [ ]:
img_input = np.expand_dims(img,axis=0)
prediction = model.predict(img_input)

plt.figure()
plt.imshow(prediction[0,:,:,0])
plt.title('Predicted Mask (Random Weights)')
plt.axis('off')
plt.show()

## Layer-by-Layer Feature Map Visualization

In [ ]:
layer_outputs = [layer.output for layer in model.layers if isinstance(layer, tf.keras.layers.Conv2D)]
feature_model = models.Model(inputs=model.input, outputs=layer_outputs)

features = feature_model.predict(img_input)

for i in range(min(4,len(features))):
    plt.figure()
    plt.imshow(features[i][0,:,:,0])
    plt.title(f'Feature Map from Conv Layer {i+1}')
    plt.axis('off')
    plt.show()